# ELT & Ingestion — Reading Files into Bronze

Goal: read CSV, JSON, Parquet and Excel files with the options real files need, write them to Bronze Delta tables, and apply the basic PySpark transformations used in the workshop.

**Learning objectives**
- Read CSV, JSON, Parquet and Excel with `spark.read` and `read_files()`, with the options real files need.
- Handle bad rows (`_corrupt_record`, `DROPMALFORMED`, `_rescued_data`) and write a Bronze Delta table.
- Use `withColumn()`, `cast()` / `try_cast()`, `when()`, `filter()`, `groupBy().agg()` and query data through views.

Prerequisites: Module 2 ([Spark Architecture](02_spark_architecture.ipynb)) and `%run ../setup/00_setup` (section 1 runs it for you).

| # | Topic | PySpark | SQL equivalent |
|---|-------|---------|----------------|
| 2 | Python you need for PySpark | variables, method chaining, `F.col()` | — |
| 3 | Medallion Architecture | — | — |
| 4 | Look at the files first | `dbutils.fs.ls()`, `spark.read.text()` | `LIST` |
| 5 | CSV | `spark.read.csv()` + options | `read_files(..., format => 'csv')` |
| 6 | JSON | `spark.read.json()`, `multiLine`, `explode()` | `read_files(..., format => 'json')` |
| 7–8 | Parquet and Excel | `spark.read.parquet()`, `format("excel")` | `read_files(...)` |
| 9 | Bad rows | `mode`, `_corrupt_record`, `rescuedDataColumn` | `read_files` adds `_rescued_data` |
| 10 | Reading files in SQL | — | `read_files()`, `schemaHints`, CTAS |
| 11 | Data exploration | `show()`, `describe()`, `summary()` | `SELECT ... LIMIT`, `DESCRIBE` |
| 12 | Writing to Delta | `saveAsTable()` | `CREATE OR REPLACE TABLE ... AS SELECT` |
| 13 | Basic transformations | `withColumn()`, `cast()`, `when()`, `filter()`, `groupBy().agg()` | `CAST`, `CASE WHEN`, `WHERE`, `GROUP BY` |
| 14 | Views and SQL | `createOrReplaceTempView()`, `spark.sql()` | `CREATE TEMP VIEW`, `CREATE VIEW` |
| ⊕ | BONUS | more column operations, filters, aggregations, joins, string functions, reader option reference | |

Data: `customers/` (CSV and Excel), `orders/` (JSON), `products/` (Parquet) and three small files in `samples/`.

> **Note:** everything after the **Summary** is ⊕ BONUS material for self-study.

## 1. Setup

Run the setup notebook. It defines the variables used below (`CATALOG`, `BRONZE_SCHEMA`, `DATASET_PATH`, ...).

In [ ]:
%run ../setup/00_setup

### Configuration

File paths for the datasets.

In [ ]:
# Folders and files in the datasets Volume (DATASET_PATH comes from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
SAMPLES_PATH = f"{DATASET_PATH}/samples"

CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{DATASET_PATH}/products/products.parquet"
EXCEL_PATH = f"{CUSTOMERS_PATH}/customers_extented.xlsx"   # the typo in the file name is intentional

In [ ]:
# All column functions in one namespace: F.col, F.when, F.sum, ...
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType

## 2. Python You Need for PySpark

You only need to *read* a few Python patterns. Each one has a SQL counterpart.

| Python | What it means | SQL analogy |
|---|---|---|
| `min_amount = 100` | A variable: a name for a value | `DECLARE VARIABLE min_amount = 100` |
| `orders = spark.read...` | A variable can hold a DataFrame: a query over files or a table, not a copy of the data | a view |
| `orders.filter(...)` | A method call: `object.method(arguments)` | a clause applied to a table |
| `(orders .filter(...) .select(...))` | Method chaining: one step per line, the whole chain wrapped in parentheses | `SELECT ... FROM ... WHERE ...` |
| `"total_amount"` vs `F.col("total_amount")` | A string is only a column name; `F.col()` is a column expression you can compute with, e.g. `F.col("total_amount") * 2` | `total_amount * 2` |
| `df = df.withColumn(...)` | A DataFrame never changes in place. Each method returns a **new** DataFrame, so assign the result to keep it | a chain of CTEs: `WITH step2 AS (SELECT ... FROM step1)` |
| `None`, `True`, `False` | Python names for `NULL`, `TRUE`, `FALSE` | |
| `# comment` | A comment | `-- comment` |

### Conditions

`==` compares (a single `=` assigns a variable). `&` means AND, `|` means OR, `~` means NOT, and each condition goes in its own parentheses:
`(F.col("state") == "NY") & (F.col("total_amount") > 100)` is `state = 'NY' AND total_amount > 100`.

In [ ]:
# Variables and a method chain
min_amount = 100                                            # a Python variable

orders_raw = spark.read.json(ORDERS_JSON)                   # a DataFrame variable

big_orders = (                                              # parentheses: one step per line
    orders_raw
    .filter(F.col("total_amount") > min_amount)             # WHERE total_amount > 100
    .select("order_id", "payment_method", "total_amount")   # SELECT order_id, payment_method, total_amount
)

In [ ]:
# A DataFrame never changes in place
big_orders.withColumn("amount_x2", F.col("total_amount") * 2)               # result discarded: big_orders is unchanged
big_orders = big_orders.withColumn("amount_x2", F.col("total_amount") * 2)  # assign the result to keep it

display(big_orders.limit(5))                                # display() is an action: now Spark runs the query

## 3. Medallion Architecture — Bronze / Silver / Gold

Before we load data, it helps to see **why** we organise it into three layers. This Lakehouse design pattern appears in every Databricks project.

![Medallion architecture overview](<../../assets/images/a433428da3144cfea92e0681ab5c73a9.png>)

### Three Layers

| Layer      | Purpose                | Transformations                  | Example                                      |
|------------|------------------------|----------------------------------|----------------------------------------------|
| **Bronze** | Raw source copy        | None or minimal                  | `customers_raw` — every row from CSV         |
| **Silver** | Cleaning & normalisation | Deduplication, types, null filtering | `customers_clean` — correct types, no duplicates |
| **Gold**   | Business / BI aggregations | GROUP BY, JOINs, metrics         | `customers_by_region` — customer count per region |

> **Note:** Delta Lake is the default format in Databricks, so every layer has built-in change history (Time Travel), transactionality and performance.

### What This Module Builds

We build the **Bronze** layer: files → Delta table plus load metadata. Section 13 previews what Silver does; the full Silver and Gold steps come in the final project.

![Medallion architecture](../../assets/images/555ecf8e271a4b4381567ef3e1b5a415.png)

## 4. Look at the Files First

Before you choose reader options, check which files you have and what a raw line looks like: the separator, the header, the date format.

SQL: `LIST '/Volumes/<catalog>/default/datasets/customers'`

In [ ]:
# Files in the customers folder: name, size, modification time
display(dbutils.fs.ls(CUSTOMERS_PATH))

In [ ]:
# Raw lines, not parsed: one string column called value
display(spark.read.text(CUSTOMERS_CSV).limit(5))

## 5. CSV

| Option | Default | What it does |
|---|---|---|
| `header` | `false` | First line holds the column names |
| `inferSchema` | `false` | Scan the data to guess the types (extra pass over the file) |
| `sep` | `,` | Column separator, e.g. `;` |
| `nullValue` | `""` | Text that means `NULL`, e.g. `N/A` |
| `dateFormat` | `yyyy-MM-dd` | Date pattern, e.g. `dd.MM.yyyy` |
| `.schema(...)` | — | Explicit column names and types; skips inference |

SQL: `read_files('<path>', format => 'csv', header => true, sep => ';', ...)` takes the same options.

In [ ]:
# Defaults: no header, every column a STRING called _c0, _c1, ...
display(spark.read.csv(CUSTOMERS_CSV).limit(3))

In [ ]:
# header + inferSchema: real column names and types
customers_auto_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_CSV)
)
customers_auto_df.printSchema()

### Explicit Schema

An explicit schema gives predictable types and skips the inference pass. Write it as a **DDL string** (the column list from `CREATE TABLE`) or as a `StructType`.

> **Warning:** the schema in the next cell is **wrong on purpose**. Run it and find the mistake.

In [ ]:
# INTENTIONALLY WRONG schema: city before email, three columns missing
wrong_schema = "customer_id STRING, first_name STRING, last_name STRING, city STRING, email STRING, registration_date DATE"
display(spark.read.option("header", True).schema(wrong_schema).csv(CUSTOMERS_CSV).limit(5))

> **Note:** CSV columns are matched **by position**, not by name. The output shows e-mails in `city` and phone numbers in `email`, and no error is raised.

In [ ]:
# Correct schema: all 10 columns in the CSV order
customer_schema_ddl = (
    "customer_id STRING, first_name STRING, last_name STRING, email STRING, phone STRING, "
    "city STRING, state STRING, country STRING, registration_date DATE, customer_segment STRING"
)

customers_df = spark.read.option("header", True).schema(customer_schema_ddl).csv(CUSTOMERS_CSV)
customers_df.printSchema()

In [ ]:
# The same schema as a StructType: you will meet it in existing code and in the final project
customer_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("email", StringType()),
    StructField("phone", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("country", StringType()),
    StructField("registration_date", DateType()),
    StructField("customer_segment", StringType()),
])

### nullValue: Text That Means NULL

The `phone` column holds the text `N/A` in 65 rows. By default it stays a string; with `nullValue` it becomes `NULL`.

In [ ]:
# Rows with phone = NULL after reading N/A as NULL. Without nullValue the count is 0
spark.read.option("header", True).option("nullValue", "N/A").csv(CUSTOMERS_CSV).filter("phone IS NULL").count()

### Several Files in One Read

A path can hold a pattern: `*` (any text), `?` (one character), `[1-2]` (a range). The hidden `_metadata` column tells you which file each row came from.

SQL: `read_files('<folder>', format => 'csv', fileNamePattern => 'customers*.csv')`

In [ ]:
# customers.csv + customers_new.csv; the .xlsx in the same folder is skipped
all_customers_df = spark.read.option("header", True).csv(f"{CUSTOMERS_PATH}/customers*.csv")
display(all_customers_df.groupBy("_metadata.file_name").count())

### A CSV From Another System

`samples/customers_pl.csv` uses `;` as the separator, Polish dates (`15.01.2025`), `N/A` for missing values and a quoted city that contains `;`.

In [ ]:
# The raw file
display(spark.read.text(f"{SAMPLES_PATH}/customers_pl.csv"))

In [ ]:
# sep, dateFormat and nullValue make it a clean table
customers_pl_df = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("dateFormat", "dd.MM.yyyy")
    .option("nullValue", "N/A")
    .schema("customer_id STRING, first_name STRING, last_name STRING, city STRING, registration_date DATE, customer_segment STRING")
    .csv(f"{SAMPLES_PATH}/customers_pl.csv")
)
display(customers_pl_df)

## 6. JSON

`spark.read.json()` reads JSON Lines: one JSON object per line. The JSON reader always infers the schema; fields are matched **by name**, so the order does not matter.

SQL: `read_files('<path>/orders_batch.json', format => 'json')`

In [ ]:
# Inferred schema
orders_auto_df = spark.read.json(ORDERS_JSON)
orders_auto_df.printSchema()

In [ ]:
# Explicit schema: only the 6 columns we need; the other fields are skipped
orders_schema = (
    "order_id STRING, customer_id STRING, order_datetime STRING, "
    "quantity INT, total_amount DOUBLE, payment_method STRING"
)
orders_df = spark.read.schema(orders_schema).json(ORDERS_JSON)
display(orders_df.limit(5))

In [ ]:
# A file pattern on the stream folder: files 001 and 002 only
stream_df = spark.read.json(f"{ORDERS_PATH}/stream/orders_stream_00[1-2].json")
display(stream_df.groupBy("_metadata.file_name").count())

### Nested, Multi-line JSON

`samples/orders_nested.json` is one JSON array spread over many lines, with a nested `customer` object and an `items` list. It needs `multiLine`; without it Spark expects one object per line and puts the lines into `_corrupt_record`.

In [ ]:
# multiLine: one JSON document across many lines
nested_df = spark.read.option("multiLine", True).json(f"{SAMPLES_PATH}/orders_nested.json")
nested_df.printSchema()
display(nested_df)

In [ ]:
# customer.id reads a nested field; explode() makes one row per item in the list
items_df = nested_df.select("order_id", "customer.id", F.explode("items").alias("item"))
display(items_df.select("order_id", "id", "item.*"))

## 7. Parquet

Parquet is the columnar file format under Delta Lake. The schema is stored in the file, so there is no `header`, no `inferSchema` and no schema to write.

SQL: `read_files('<path>/products.parquet')`

In [ ]:
# Types come from the file
products_df = spark.read.parquet(PRODUCTS_PARQUET)
products_df.printSchema()
display(products_df.limit(5))

## 8. Excel

The built-in reader `format("excel")` reads `.xlsx` and `.xls` (Databricks Runtime 17.1 and above, also serverless). `dataAddress` selects the sheet and the cell range; `headerRows` (0 or 1) says whether the first row of the range holds column names.

SQL: `read_files('<path>.xlsx', format => 'excel', dataAddress => 'Arkusz1!D6:M26', headerRows => 1)`

In [ ]:
# Which sheets does the file have?
display(spark.read.format("excel").option("operation", "listSheets").load(EXCEL_PATH))

In [ ]:
# Without options: empty columns A–C come as _c0.._c2 and the header row is read as data
display(spark.read.format("excel").load(EXCEL_PATH))

In [ ]:
# The table starts in D6: read only that range, first row = column names
customers_extended_df = (
    spark.read
    .format("excel")
    .option("dataAddress", "Arkusz1!D6:M26")
    .option("headerRows", 1)
    .load(EXCEL_PATH)
)
display(customers_extended_df)

Look at `phone`: numbers come with thousand separators, and cells with a formula (`=1-555-123-4567`) come in as the computed value (`-5244`). Excel data needs cleaning in Silver.

## 9. Bad Rows

`samples/orders_bad.csv` has one row with the text `three` in `quantity` and one cut-off row.

| Mode / option | What happens to a bad row |
|---|---|
| `PERMISSIVE` (default) | Kept; fields that do not fit become `NULL`; the raw line goes to `_corrupt_record` if that column is in the schema |
| `DROPMALFORMED` | Dropped |
| `FAILFAST` | The read fails |
| `rescuedDataColumn` | A value of the wrong type is saved as JSON in `_rescued_data` instead of being lost |

In [ ]:
# Schema used by the cells below
bad_schema = "order_id STRING, customer_id STRING, quantity INT, total_amount DOUBLE, order_date DATE"
BAD_CSV = f"{SAMPLES_PATH}/orders_bad.csv"

In [ ]:
# PERMISSIVE: all 5 rows stay; the raw line of each bad row is in _corrupt_record
display(spark.read.option("header", True).schema(bad_schema + ", _corrupt_record STRING").csv(BAD_CSV))

In [ ]:
# DROPMALFORMED: the 2 bad rows are gone
# .option("mode", "FAILFAST") would stop the read at the first bad row
display(spark.read.option("header", True).option("mode", "DROPMALFORMED").schema(bad_schema).csv(BAD_CSV))

In [ ]:
# rescuedDataColumn: "three" is kept in _rescued_data, quantity is NULL
display(spark.read.option("header", True).option("rescuedDataColumn", "_rescued_data").schema(bad_schema).csv(BAD_CSV))

## 10. read_files() — Reading Files in SQL

`read_files(path, format => ...)` reads files in a SQL query. Its defaults differ from `spark.read`: for CSV it uses the header and infers the types, and it always adds the `_rescued_data` column.

> **Tip:** `read_files()` is the recommended way to read files in SQL workflows with Unity Catalog.

In [ ]:
# Make DATASET_PATH available to the %sql cells as the session variable dataset_path
spark.sql(f"DECLARE OR REPLACE VARIABLE dataset_path STRING DEFAULT '{DATASET_PATH}'")

In [ ]:
%sql
-- Header and types without any option; note the _rescued_data column at the end
SELECT * FROM read_files(dataset_path || '/customers/customers.csv', format => 'csv')
LIMIT 5

In [ ]:
%sql
-- schemaHints: set the type of chosen columns, infer the rest
SELECT order_id, order_datetime, quantity
FROM read_files(dataset_path || '/orders/orders_batch.json',
                format => 'json',
                schemaHints => 'order_datetime TIMESTAMP, quantity INT')
LIMIT 5

In [ ]:
%sql
-- Bronze table straight from the files: raw columns + load metadata (CTAS)
CREATE OR REPLACE TABLE bronze.orders AS
SELECT *,
       current_timestamp() AS _load_ts,
       _metadata.file_path AS _source_file
FROM read_files(dataset_path || '/orders/orders_batch.json', format => 'json')

In [ ]:
%sql
SELECT * FROM bronze.orders LIMIT 5

## 11. Data Exploration — `show()`, `describe()`, `summary()`

| Method | What it returns | SQL equivalent |
|---|---|---|
| `display(df)` | Interactive table with charts and Data Profile (Databricks only) | `SELECT * FROM t` in a `%sql` cell |
| `df.show(n, truncate)` | Plain-text preview of `n` rows; works in jobs and logs too | `SELECT * FROM t LIMIT n` |
| `df.printSchema()` | Column names and types | `DESCRIBE TABLE t` |
| `df.describe(*cols)` | `count`, `mean`, `stddev`, `min`, `max` | `SELECT count(c), avg(c), stddev(c), min(c), max(c) FROM t` |
| `df.summary(*stats)` | `describe()` plus percentiles `25%`, `50%`, `75%`; you can pick the statistics | `percentile(c, 0.5)` |
| `df.count()`, `df.select(c).distinct().count()` | Number of rows / distinct values | `SELECT count(*)`, `count(DISTINCT c)` |

In [ ]:
# show(): plain-text preview. SQL: SELECT * FROM customers LIMIT 5
customers_df.show(5, truncate=False)

In [ ]:
# Row count. SQL: SELECT count(*) FROM customers
customers_df.count()

In [ ]:
# describe() on numeric columns. SQL: SELECT count(x), avg(x), stddev(x), min(x), max(x) FROM orders
display(orders_df.describe("quantity", "total_amount"))

In [ ]:
# summary() with chosen statistics
display(orders_df.select("quantity", "total_amount").summary("min", "25%", "50%", "75%", "max"))

> **Note:** the minimum `total_amount` is negative (returns or broken rows). Bronze keeps such rows as they are; Silver decides what to do with them.

## 12. Write to Delta — PySpark

The customers Bronze table, built from a DataFrame. You use this pattern in the workshop and in the final project.

| Step | PySpark | SQL equivalent (section 10) |
|---|---|---|
| Load timestamp | `.withColumn("_load_ts", F.current_timestamp())` | `current_timestamp() AS _load_ts` |
| Source file | `.withColumn("_source_file", F.col("_metadata.file_path"))` | `_metadata.file_path AS _source_file` |
| Save as a managed Delta table | `.write.mode("overwrite").saveAsTable("catalog.schema.table")` | `CREATE OR REPLACE TABLE ... AS SELECT` |

In [ ]:
# Raw data + load metadata: no business logic, no filtering
BRONZE_CUSTOMERS = f"{CATALOG}.{BRONZE_SCHEMA}.customers"

customers_bronze = (
    customers_df
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [ ]:
# Write as a managed Delta table (Delta is the default format)
customers_bronze.write.mode("overwrite").saveAsTable(BRONZE_CUSTOMERS)
display(spark.table(BRONZE_CUSTOMERS).limit(5))

> **Tip:** open **Catalog** → your catalog → `bronze` → `customers` to see the table, its schema and its history.

## 13. Basic Transformations

Each transformation returns a **new** DataFrame. These are the patterns used in the workshop and in the Silver and Gold steps of the final project.

| PySpark | SQL |
|---|---|
| `withColumn("name", expr)` | `SELECT *, expr AS name` |
| `F.col("x").cast("date")` | `CAST(x AS DATE)` |
| `F.when(c, v).otherwise(v2)` | `CASE WHEN c THEN v ELSE v2 END` |
| `filter(cond)` | `WHERE cond` |
| `groupBy("x").agg(F.sum("y").alias("s"))` | `SELECT x, SUM(y) AS s ... GROUP BY x` |

In [ ]:
# withColumn adds a column. concat_ws joins strings and skips NULL
# SQL: SELECT *, concat_ws(' ', first_name, last_name) AS full_name, year(registration_date) AS registration_year
customers_enriched = (
    customers_df
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("registration_year", F.year(F.col("registration_date")))
)
display(customers_enriched.select("customer_id", "full_name", "registration_year").limit(5))

ANSI mode (strict SQL-standard behaviour) is on in Databricks Runtime 17.0 and above and on serverless compute. A value that cannot be converted makes the query fail with `CAST_INVALID_INPUT` instead of becoming `NULL`. When bad values are expected, use `try_cast`, which returns `NULL` for them.

| | PySpark | SQL |
|---|---|---|
| Strict: fails on a bad value | `F.col("x").cast("double")` | `CAST(x AS DOUBLE)` |
| Tolerant: `NULL` for a bad value | `F.expr("try_cast(x AS DOUBLE)")` | `try_cast(x AS DOUBLE)` |

In [ ]:
# Without a schema every CSV column is a STRING
customers_raw_df = spark.read.option("header", True).csv(CUSTOMERS_CSV)

In [ ]:
# STRING -> TIMESTAMP. SQL: CAST(order_datetime AS TIMESTAMP)
orders_ts = orders_df.withColumn("order_ts", F.col("order_datetime").cast("timestamp"))
display(orders_ts.select("order_id", "order_datetime", "order_ts").limit(5))

In [ ]:
# phone holds text like "+48 694 026 542": CAST fails with CAST_INVALID_INPUT (ANSI mode)
try:
    customers_raw_df.select(F.col("phone").cast("double")).limit(5).collect()
except Exception as e:
    print(str(e).splitlines()[0])

In [ ]:
# try_cast returns NULL instead of failing. SQL: try_cast(phone AS DOUBLE)
display(customers_raw_df.select("phone", F.expr("try_cast(phone AS DOUBLE)").alias("phone_as_double")).limit(5))

In [ ]:
# when / otherwise = CASE WHEN
customers_classified = customers_df.withColumn(
    "customer_tier",
    F.when(F.year("registration_date") >= 2025, "New")
    .when(F.year("registration_date") >= 2024, "Regular")
    .otherwise("Loyal")
)
display(customers_classified.groupBy("customer_tier").count())

In [ ]:
# filter = WHERE. Each condition in parentheses; & = AND, | = OR
# SQL: WHERE email IS NOT NULL AND state = 'NY'
ny_customers = customers_df.filter(F.col("email").isNotNull() & (F.col("state") == "NY"))
display(ny_customers.limit(5))

In [ ]:
# groupBy + agg = GROUP BY. SQL: SELECT payment_method, SUM(total_amount) AS total_revenue, ... GROUP BY payment_method
revenue_by_payment = (
    orders_df
    .groupBy("payment_method")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value"),
        F.count("*").alias("order_count"),
    )
    .orderBy(F.desc("total_revenue"))
)
display(revenue_by_payment)

## 14. Views and SQL

A view gives a query a name you can use in SQL. It stores the query, not a copy of the data.

| View type | Visible in | Lives until | Create |
|---|---|---|---|
| **Temp view** | This notebook (session) | The session ends | `df.createOrReplaceTempView("name")` / `CREATE TEMP VIEW` |
| **Global temp view** | All notebooks on the same classic cluster | The cluster restarts | `df.createOrReplaceGlobalTempView("name")`, read as `global_temp.name` |
| **View** | Everyone with access, in Unity Catalog | You drop it | `CREATE VIEW catalog.schema.name AS SELECT ...` |

In [ ]:
# Temp views: names for our DataFrames, usable in SQL in this notebook
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")

In [ ]:
# SQL on a temp view from Python: spark.sql() returns a DataFrame
display(spark.sql("SELECT state, COUNT(*) AS customers FROM customers GROUP BY state ORDER BY customers DESC LIMIT 5"))

In [ ]:
%sql
-- The same view in a %sql cell
SELECT payment_method, COUNT(*) AS orders
FROM orders
GROUP BY payment_method

### Global Temp View

Visible to every notebook attached to the same **classic** cluster. Serverless compute does not support global temp views; skip the next two cells there.

In [ ]:
# Register a global temp view
customers_df.createOrReplaceGlobalTempView("customers_global")

In [ ]:
%sql
-- Read it through the global_temp schema (also from another notebook on this cluster)
SELECT customer_segment, COUNT(*) AS customers
FROM global_temp.customers_global
GROUP BY customer_segment

### View in Unity Catalog

A permanent view lives in your catalog next to the tables. Other notebooks, the SQL Editor and dashboards can use it. Open **Catalog** → your catalog → `bronze` to see it.

In [ ]:
%sql
-- A permanent view over the Bronze table
CREATE OR REPLACE VIEW bronze.v_customers_ny AS
SELECT customer_id, first_name, last_name, city
FROM bronze.customers
WHERE state = 'NY'

In [ ]:
%sql
SELECT * FROM bronze.v_customers_ny LIMIT 5

## Summary

- **Look first:** `dbutils.fs.ls()` and `spark.read.text()` show the files and a raw line before you choose options.
- **CSV:** `header`, `inferSchema`, `sep`, `nullValue`, `dateFormat`; an explicit schema gives predictable types; CSV columns match by position, JSON fields by name.
- **Many files and formats:** file patterns (`customers*.csv`) with `_metadata.file_name`; `multiLine` + `explode()` for nested JSON; Parquet carries its own schema; Excel needs `dataAddress` and `headerRows`.
- **Bad rows:** `PERMISSIVE` + `_corrupt_record`, `DROPMALFORMED`, `FAILFAST`; `_rescued_data` keeps values of the wrong type.
- **SQL:** `read_files()` infers the header and types and adds `_rescued_data`; `schemaHints` fixes chosen columns; CTAS writes Bronze.
- **Bronze in PySpark:** raw data + `_load_ts` + `_source_file`, written with `saveAsTable`.
- **Transform:** `withColumn`, `cast` (ANSI: a bad value fails, `try_cast` returns `NULL`), `when`, `filter`, `groupBy().agg()`.
- **Views:** temp view (this notebook), global temp view (this classic cluster), view in Unity Catalog (permanent).

Next: [Delta Lake](04_delta_fundamentals.ipynb) (Module 4).

---
> ⊕ **BONUS — More Column and Row Operations**
>
> select, withColumnRenamed, drop, distinct, orderBy and isin map 1:1 to SQL you know.
---

## B1. More Column and Row Operations

### select — Choosing Columns

`df.select(...)` keeps only the listed columns or expressions.

SQL: `SELECT customer_id, upper(first_name) AS first_name_upper, email FROM customers`

In [ ]:
# Select specific columns
customers_selected = customers_df.select("customer_id", "first_name", "last_name", "email")
display(customers_selected.limit(5))

In [ ]:
# Select with column expressions
customers_transformed = customers_df.select(
    F.col("customer_id"),
    F.upper(F.col("first_name")).alias("first_name_upper"),   # .alias() = AS
    F.col("email")
)
display(customers_transformed.limit(5))

### withColumnRenamed — Renaming

`df.withColumnRenamed("old", "new")` renames one column. To rename several, use `select()` with `.alias()`.

SQL: `SELECT customer_id AS id, ... FROM customers`

In [ ]:
# Rename single column
customers_renamed = customers_df.withColumnRenamed("customer_id", "id")
display(customers_renamed.limit(5))

In [ ]:
# Rename multiple columns using select with alias
customers_multi_renamed = customers_df.select(
    F.col("customer_id").alias("id"),
    F.col("first_name").alias("fname"),
    F.col("last_name").alias("lname"),
    F.col("email"),
    F.col("city")
)
display(customers_multi_renamed.limit(5))

### drop — Removing Columns

`df.drop("a", "b")` returns the DataFrame without the listed columns; a name that does not exist is ignored.

SQL: `SELECT * EXCEPT (email, city, country) FROM customers`

In [ ]:
# Drop columns. SQL: SELECT * EXCEPT (email, city, country)
customers_minimal = customers_df.drop("email", "city", "country")
display(customers_minimal.limit(5))

### distinct / dropDuplicates — Unique Rows

`df.distinct()` keeps unique rows (all columns compared); `df.dropDuplicates(["a", "b"])` compares only the listed columns.

SQL: `SELECT DISTINCT state FROM customers`

In [ ]:
# Get distinct states (every customer in this dataset is from the USA, so country has one value)
distinct_states = customers_df.select("state").distinct()
display(distinct_states.orderBy("state"))

In [ ]:
# Drop duplicates based on specific columns
# Raw data: note the mixed spellings (California / CALIFORNIA, CA / ca) — cleaning belongs to Silver
unique_locations = customers_df.select("city", "state").dropDuplicates(["city", "state"])
display(unique_locations.orderBy("state", "city").limit(10))

### orderBy — Sorting

`df.orderBy(...)` (alias `sort()`) sorts rows, ascending by default; use `F.desc("col")` for descending. A global sort moves data between machines, so sort as late as possible.

SQL: `SELECT * FROM customers ORDER BY country ASC, registration_date DESC`

In [ ]:
# Sort by single column (ascending)
customers_sorted_asc = customers_df.orderBy("registration_date")
display(customers_sorted_asc.select("customer_id", "first_name", "last_name", "registration_date").limit(5))

In [ ]:
# Sort by multiple columns with different directions
customers_sorted_multi = customers_df.orderBy(F.asc("country"), F.desc("registration_date"))
display(customers_sorted_multi.select("customer_id", "first_name", "country", "registration_date").limit(10))

### isin — Filter by List

`F.col("x").isin([...])` is true when the value is in the list; negate it with `~`.

SQL: `SELECT * FROM customers WHERE state IN ('NY', 'CA', 'TX', 'FL')`

In [ ]:
# Filter by list of states
selected_states = ["NY", "CA", "TX", "FL"]
customers_selected_states = customers_df.filter(F.col("state").isin(selected_states))

# Show distribution by state
display(customers_selected_states.groupBy("state").count().orderBy(F.desc("count")))

---
> ⊕ **BONUS — More Filters**
>
> where() with a SQL string, OR conditions and IS NULL.
---

## B2. More Filters

`df.filter(condition)` keeps the rows that meet the condition. `where()` is the same method under its SQL name.

In [ ]:
# where() is the same as filter(); a SQL condition string also works
nyc_customers = customers_df.where("city = 'New York'")
display(nyc_customers.select("customer_id", "first_name", "city", "state").limit(5))

In [ ]:
# OR condition. SQL: WHERE state = 'NY' OR state = 'CA'
ny_or_ca = customers_df.filter(
    (F.col("state") == "NY") | (F.col("state") == "CA")
)
display(ny_or_ca.groupBy("state").count())

In [ ]:
# Filter rows where city IS null. SQL: WHERE city IS NULL
display(customers_df.filter(F.col("city").isNull()).select("customer_id", "first_name", "city").limit(5))

In [ ]:
# Replace NULL with a default value. SQL: coalesce(city, 'Unknown')
display(customers_df.withColumn("city_safe", F.coalesce(F.col("city"), F.lit("Unknown"))).filter(F.col("city").isNull()).limit(5))

---
> ⊕ **BONUS — fillna / dropna — Replacing and Dropping Nulls**
>
> Silver-layer data quality patterns.
---

## B3. fillna / dropna — Handling Missing Values

- `DataFrame.fillna(value, subset=None)` (alias: `na.fill()`) replaces `null` values with a given value. It can take a dict to set a different fill value per column. It works on `StringType`, numeric types and `BooleanType`.
- `DataFrame.dropna(how='any', thresh=None, subset=None)` (alias: `na.drop()`) removes rows that contain null values:
  - `how='any'` — drop if **any** specified column is null
  - `how='all'` — drop only if **all** specified columns are null
  - `thresh=N` — keep rows with at least N non-null values

| Method | Example | Effect |
|---|---|---|
| `fillna("Unknown")` | All nulls in string cols → `"Unknown"` | Global fill |
| `fillna(0, subset=["amount"])` | Nulls in `amount` → `0` | Column-specific |
| `fillna({"city": "Unknown", "age": 0})` | Per-column dict | Mixed types |
| `dropna()` | Drop rows with any null | Strict |
| `dropna(how="all")` | Drop rows where all are null | Lenient |
| `dropna(subset=["customer_id"])` | Only check these columns | Targeted |

---
> ⊕ **BONUS — String Operations**
>
> `like`, `contains` and `startswith` are simple string filters, the same as SQL `LIKE`.
---

## B4. String Operations (`like`, `contains`, `startswith`)

- `Column.like(pattern)` matches a SQL wildcard pattern: `%` (any string), `_` (one character). SQL equivalent: `LIKE`.
- `Column.contains(other)` returns `true` if the column contains the substring `other`.
- `Column.startswith(other)` / `endswith(other)` check the prefix / suffix.
- `Column.rlike(pattern)` matches a regular expression (Java regex).

All string operations return `null` if the column value is `null`.

In [ ]:
# Filter using like (SQL-style pattern matching). SQL: WHERE email LIKE '%@gmail.com'
gmail_customers = customers_df.filter(F.col("email").like("%@gmail.com"))
display(gmail_customers.select("customer_id", "first_name", "email").limit(5))

In [ ]:
# Filter using contains. SQL: WHERE city LIKE '%New%'
new_cities = customers_df.filter(F.col("city").contains("New"))
display(new_cities.select("customer_id", "first_name", "city").limit(5))

In [ ]:
# Filter using startswith. SQL: WHERE first_name LIKE 'J%'
j_names = customers_df.filter(F.col("first_name").startswith("J"))
display(j_names.select("customer_id", "first_name", "last_name").limit(5))

---
> ⊕ **BONUS — More Aggregations: count per group, min / max, HAVING**
>
> Variations of the `groupBy().agg()` pattern from section 13.
---

## B5. More Aggregations

In [ ]:
# Count by state. SQL: SELECT state, COUNT(*) AS count FROM customers GROUP BY state ORDER BY count DESC
# Raw data: lower-case codes (ny, nj) form their own groups; Silver would standardise them
customers_by_state = customers_df.groupBy("state").count().orderBy(F.desc("count"))
display(customers_by_state.limit(10))

### min / max

`F.min(col)` / `F.max(col)` return the smallest / largest value; `NULL` values are ignored. Without `groupBy`, `df.agg(...)` computes them over the whole DataFrame.

SQL: `SELECT min(total_amount), max(total_amount), avg(total_amount) FROM orders`

In [ ]:
# Min and max order amounts
# Negative amounts are returns or broken rows in the raw data — Silver decides how to treat them
order_stats = orders_df.agg(
    F.min("total_amount").alias("min_amount"),
    F.max("total_amount").alias("max_amount"),
    F.avg("total_amount").alias("avg_amount")
)
display(order_stats)

### Multiple Aggregations with agg()

One `agg(...)` call can compute many aggregates in a single pass over the data.

SQL: `SELECT customer_id, COUNT(*), SUM(total_amount), AVG(total_amount), MIN(total_amount), MAX(total_amount) FROM orders GROUP BY customer_id`

In [ ]:
# Example: Multiple aggregations on orders by customer
# The top rows are CUST999999 (placeholder for an unknown customer) and NULL — raw-data issues, not real customers
customer_stats = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("total_amount").alias("total_spent"),
        F.avg("total_amount").alias("avg_order_value"),
        F.min("total_amount").alias("min_order"),
        F.max("total_amount").alias("max_order")
    )
    .orderBy(F.desc("total_spent"))
)

display(customer_stats.limit(10))

### HAVING Equivalent (Filter after groupBy)

`filter()` **before** `groupBy` is `WHERE`; `filter()` **after** `groupBy().agg()` is `HAVING`.

SQL: `SELECT customer_id, COUNT(*) AS order_count FROM orders GROUP BY customer_id HAVING COUNT(*) > 5`

In [ ]:
# Customers with more than 5 orders (CUST999999 and NULL again lead — see the note above)
high_frequency = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("total_amount").alias("total_spent")
    )
    .filter(F.col("order_count") > 5)
    .orderBy(F.desc("order_count"))
)

display(high_frequency.limit(10))

---
> ⊕ **BONUS — Joining DataFrames**
>
> You know JOIN from SQL; this section prepares the optional Workshop Task 12, which checks the row-duplication effect shown here.
---

## B6. Joining DataFrames

`df.join(other, condition, how)` combines two DataFrames; `how` is `"inner"` (default), `"left"`, `"right"` or `"full"`.

SQL: `SELECT o.order_id, o.customer_id, c.first_name, c.country, o.total_amount FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id`

> **Warning:** when both DataFrames have a column with the same name (here `customer_id`), refer to it through its DataFrame — `orders_df["customer_id"]` — both in the join condition and in `select()`. Otherwise Spark reports an *ambiguous reference*.

In [ ]:
# Left join: keep every order, attach customer attributes where customer_id matches
orders_with_customers = (
    orders_df
    .join(customers_df, orders_df["customer_id"] == customers_df["customer_id"], "left")
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        customers_df["first_name"],
        customers_df["country"],
        orders_df["total_amount"],
    )
)

display(orders_with_customers.limit(10))

In [ ]:
# Joined DataFrame works like any other — aggregate it
revenue_by_country = (
    orders_with_customers
    .groupBy("country")
    .agg(F.count("*").alias("order_count"), F.sum("total_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_country)

> **Note:** why is `order_count` higher than the number of orders?
>
> `orders_df` has **100,000** orders, but the join returns **more** rows. `customers_df` contains duplicate `customer_id` values
> (two different people share one ID), so every order of such a customer is matched **twice** — and its `total_amount` is counted twice.

A join multiplies rows whenever the key on the right side is not unique. Check it with
`customers_df.groupBy("customer_id").count().filter("count > 1")` — and deduplicate (`dropDuplicates(["customer_id"])`) before joining in Silver.
Workshop Task 12 checks for exactly this effect.

---
> ⊕ **BONUS — DataFrame API vs SQL**
>
> The same query written both ways; both run on the same engine.
---

## B7. DataFrame API vs SQL — Same Result

Both approaches produce the same result: the same Catalyst optimizer plans them.

In [ ]:
# Same query, two approaches: first the DataFrame API
df_api_result = (
    customers_df
    .groupBy("city")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)

display(df_api_result)

In [ ]:
# Same query in SQL
sql_result = spark.sql("""
    SELECT
        city,
        COUNT(*) as count
    FROM customers
    GROUP BY city
    ORDER BY count DESC
    LIMIT 5
""")

display(sql_result)

> **Note:** `customer_id` is not unique in the raw customers file: two people can share one ID, so such an ID appears twice below with the same totals. Silver would deduplicate first.

In [ ]:
# JOIN + GROUP BY + HAVING in one SQL query
display(spark.sql("""
    SELECT c.customer_id, c.first_name, COUNT(o.order_id) AS total_orders, ROUND(SUM(o.total_amount), 2) AS total_spent
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.first_name
    HAVING COUNT(o.order_id) > 0
    ORDER BY total_spent DESC
    LIMIT 10
"""))

### SQL vs DataFrame API

| Criteria | SQL | DataFrame API |
|---|---|---|
| **Best for** | JOINs, ad-hoc analysis, reporting | Pipelines with reusable logic |
| **Team** | SQL-first, analysts | Engineers, ML |
| **Performance** | Same engine and optimizer | Same engine and optimizer |

---
> ⊕ **BONUS — inferSchema vs Explicit Schema: Read Time**
>
> inferSchema scans the file twice; the cost matters only for large files.
---

## B8. inferSchema vs Explicit Schema: Read Time

inferSchema reads the file twice. Run both cells and compare the run time shown under each cell.

In [ ]:
# inferSchema: one extra pass over the file to guess the types
spark.read.option("header", True).option("inferSchema", True).csv(CUSTOMERS_CSV).count()

In [ ]:
# Explicit schema: no inference pass
spark.read.option("header", True).schema(customer_schema_ddl).csv(CUSTOMERS_CSV).count()

---
> ⊕ **BONUS — Reader Option Reference**
>
> Full lists of CSV and JSON options for production files.
---

## B9. CSV Reader Options

The CSV options you need for production files:

| Option | Type | Default | Description |
|---|---|---|---|
| `header` | bool | `false` | First row as column names |
| `inferSchema` | bool | `false` | Auto-detect types (2 scans — expensive!) |
| `delimiter` / `sep` | string | `,` | Column separator (e.g. `;`, `\t`) |
| `quote` | string | `"` | Quote character for text fields |
| `escape` | string | `\` | Escape character inside quoted strings |
| `nullValue` | string | `""` | String to treat as `null` (e.g. `"N/A"`) |
| `emptyValue` | string | `""` | Value to replace empty strings |
| `nanValue` | string | `NaN` | String to treat as `NaN` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for `DateType` columns |
| `timestampFormat` | string | ISO 8601 | Pattern for `TimestampType` columns |
| `encoding` | string | `UTF-8` | File encoding (e.g. `ISO-8859-1`, `UTF-16`) |
| `multiLine` | bool | `false` | Allow newlines inside quoted fields |
| `comment` | string | disabled | Lines starting with this char are skipped |
| `ignoreLeadingWhiteSpace` | bool | `false` | Strip leading whitespace |
| `ignoreTrailingWhiteSpace` | bool | `false` | Strip trailing whitespace |
| `mode` | string | `PERMISSIVE` | Error handling: `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Column to store malformed rows (PERMISSIVE mode) |
| `maxColumns` | int | `20480` | Max number of columns allowed |
| `samplingRatio` | double | `1.0` | Fraction of rows to scan for `inferSchema` |

## B10. JSON Reader Options

| Option | Type | Default | Description |
|---|---|---|---|
| `multiLine` | bool | `false` | Allow JSON objects spanning multiple lines |
| `allowComments` | bool | `false` | Allow `//` and `/* */` comments in JSON |
| `allowUnquotedFieldNames` | bool | `false` | Allow keys without quotes |
| `allowSingleQuotes` | bool | `true` | Allow single-quoted strings |
| `allowNumericLeadingZeros` | bool | `false` | Allow `007` as integer |
| `allowNonNumericNumbers` | bool | `true` | Allow `NaN`, `INF`, `-INF` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for date parsing |
| `timestampFormat` | string | ISO 8601 | Pattern for timestamp parsing |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Store unparseable rows |
| `mode` | string | `PERMISSIVE` | `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `primitivesAsString` | bool | `false` | Read all primitives as strings |
| `dropFieldIfAllNull` | bool | `false` | Drop columns that are all null |
| `encoding` | string | `UTF-8` | File encoding |
| `samplingRatio` | double | `1.0` | Fraction of rows used to infer the schema |